# Bibliotecas

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

import tensorflow as tf
import ee
import numpy as np
from osgeo import ogr
from osgeo import gdal
import glob,pygeoj
import rasterio
from rasterio.plot import show
from patchify import patchify, unpatchify


import json
import math
import logging
import time
from datetime import datetime, date, timedelta

import folium
from PIL import Image
from matplotlib import pyplot as plt
from IPython.display import Image

logging.getLogger('googleapicliet.discovery_cache').setLevel(logging.ERROR)

gpu_dict    = {'4090':{'GPU_AFFINTY' : 0, 'GPU_MEMORY_LIMIT_GB':12}}
sel_gpu     = '4090'
GPU_AFFINTY = gpu_dict[sel_gpu]['GPU_AFFINTY'] 
GPU_MEMORY_LIMIT_GB = gpu_dict[sel_gpu]['GPU_MEMORY_LIMIT_GB']
USER_EE_PROJECT='USER_PROJECT_ID'

try:
    ee.Initialize(project = USER_EE_PROJECT)
except:
    ee.Authenticate()
    ee.Initialize(project = USER_EE_PROJECT)

In [ ]:
EE_TILES = 'https://earthengine.googleapis.com/map/{mapid}/{{z}}/{{x}}/{{y}}?token={token}'

print('Tensorflow Version:',tf.__version__)
print('Folium Version:',folium.__version__)

gpus = tf.config.list_physical_devices('GPU')

if gpus:
  try:
    tf.config.set_visible_devices(gpus[GPU_AFFINTY], 'GPU')
    GPU_MEMORY_LIMIT_GB = GPU_MEMORY_LIMIT_GB * 1e3
    if GPU_MEMORY_LIMIT_GB == 0:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    else:
        tf.config.set_logical_device_configuration(gpus[GPU_AFFINTY],[tf.config.LogicalDeviceConfiguration(memory_limit=GPU_MEMORY_LIMIT_GB)])
    logical_gpus = tf.config.list_logical_devices('GPU')
    print(len(gpus), "Physical GPUs,", len(logical_gpus), "Logical GPUs")
  except RuntimeError as e:
    print(e)

In [3]:
def create_directory(new_folder):
  ''' Check if any of the specified folder already exist'''
  if not os.path.exists(new_folder):
      print(f'lets make the directory: {new_folder}')
      os.makedirs(new_folder)
  else: return

  def check_file_exists(paths):
    """Check if any of the specified paths already exist"""
    for path in paths:
        if os.path.exists(path):
            return True
    return False

# ENV Configs

In [4]:
# General 
GPU_AFFINITY = gpus[GPU_AFFINTY].name
print('GPU',GPU_AFFINITY)
MAPBIOMAS_V    = '10' 

# FROM SCREATCH
# VERSION        = '2_CONTINENTAL'
VERSION        = '1_CONTINENTAL_512'
MOSAIC_VERSION = '2'
SAMPLE_VERSION = '1_CONTINENTAL_no_RioGrandeDoSul_512'

SATELLITE = 'Sentinel'
match SATELLITE:
    case 'Sentinel': 
        SCALE = 10
    case 'Planet':
        SCALE = 5
    case _: #default
        SCALE = 30

GOAL_CLASS     = 'aquaculture'
GDRIVE         = f'mb{MAPBIOMAS_V}-unet-aquaculture-brazil-s2_'+VERSION

FOLDER_TRAIN   = f'mb{MAPBIOMAS_V}_aquaculture_training_samples-s2'
FOLDER_TEST    = f'mb{MAPBIOMAS_V}_aquaculture_eval_samples-s2'

TRAINING_BASE  = 'training_patches_'+MOSAIC_VERSION+'_v'+SAMPLE_VERSION
EVAL_BASE      = 'eval_patches_'+MOSAIC_VERSION+'_v'+SAMPLE_VERSION

#Local paths
CURRENT_LOCAL_PATH  = f'~/Mapbiomas/modelos/mb{MAPBIOMAS_V}-unet-{GOAL_CLASS}-s2' 

MODEL_DIR   = f'{CURRENT_LOCAL_PATH}/checkpoint/v{VERSION}'
OUTPUT_PATH = CURRENT_LOCAL_PATH+'/output/v'+VERSION
TRAIN_PATH  = CURRENT_LOCAL_PATH+'/train/'
EVAL_PATH   = CURRENT_LOCAL_PATH+'/eval/'

creatDirectory(MODEL_DIR)
creatDirectory(OUTPUT_PATH)
creatDirectory(TRAIN_PATH)
creatDirectory(EVAL_PATH)


# Specify inputs (SATELLITE bands) to the model and the response variable.
opticalBands   = ['green','red','nir','swir1']
opticalIndices = ['NDVI','MNDWI']
BANDS          = opticalBands + opticalIndices

RESPONSE = 'supervised'
FEATURES = BANDS + [RESPONSE]

# Specify the size and shape of patches expected by the model.
KERNEL_SIZE  = 512
KERNEL_SHAPE = [KERNEL_SIZE, KERNEL_SIZE]
COLUMNS = [
  tf.io.FixedLenFeature(shape=KERNEL_SHAPE, dtype=tf.float32) for k in FEATURES
]
FEATURES_DICT = dict(zip(FEATURES, COLUMNS))

# Sizes of the training and evaluation datasets.
TRAIN_SIZE = 0
EVAL_SIZE  = 0


BATCH_SIZE = 2
DROPOUT   = 0.3 #0.5
BUFFER_SIZE = 1000
OPTIMIZER = 'Nadam' 
LOSS      = 'BinaryCrossentropy'
METRICS   = ['RootMeanSquaredError', 'BinaryIoU']

GPU /physical_device:GPU:0
['green', 'red', 'nir', 'swir1', 'NDVI', 'MNDWI']


# MB10_v0_no_RioGrandeDoSul

In [ ]:
''' MB10 '''
mosaic_year   = 2023
label_version = '1'
supervisedImg_NO_RioGrandeDoSul = ee.Image('projects/solved-mb10/assets/SENTINEL2/AQUACULTURE/Supervised_mask/'+str(mosaic_year)+'-'+label_version+'-supervised_mask-no-RS').gte(1).unmask(0).rename(RESPONSE);

supervisedChannel = supervisedImg_NO_RioGrandeDoSul.toByte().rename(RESPONSE);
image  = ee.ImageCollection('projects/'+userEEProject+'/assets/'+userPATH).filter(ee.Filter.And(ee.Filter.eq('year',mosaic_year),ee.Filter.eq('mosaic',1))).mosaic().addBands(supervisedChannel)
mapid = image.getMapId({'bands': ['swir1', 'nir', 'red'], 'min': 30, 'max': 150})


# v1 CONTINENTAL (NO RIO GRANDE SO SUL)
not_RS_box = ee.Geometry.Polygon(\
        [[[-54.645333367433516, -27.262649400216283],
          [-55.809884148683516, -27.982965194198066],
          [-57.886300164308516, -30.238389059037498],
          [-56.139473992433516, -31.735840258888334],
          [-53.085274773683516, -34.18746069891844],
          [-49.624581414308516, -30.994761804272677],
          [-49.525704461183516, -29.6099620636833],
          [-49.943184929933516, -29.140850213061157],
          [-49.635567742433516, -28.380006548470092],
          [-51.195626336183516, -27.633133270802485],
          [-52.118477898683516, -27.125842508347343],
          [-53.898263054933516, -27.086724025214767]]])
trainingPolys_mb10_FULL = ee.FeatureCollection('projects/solved-mb10/assets/public/LANDSAT/AQUACULTURE/geom_train_TF'+4+'_CONTINENTAL_FULL_CONT_LITORAL')
evalPolys_mb10_FULL     = ee.FeatureCollection('projects/solved-mb10/assets/public/LANDSAT/AQUACULTURE/geom_eval_TF'+4+'_CONTINENTAL_FULL_CONT_LITORAL')

trainingPolys_no_RS = trainingPolys_mb10_FULL.filter(ee.Filter.bounds(not_RS_box).Not())
evalPolys_no_RS     = evalPolys_mb10_FULL.filter(ee.Filter.bounds(not_RS_box).Not())


polyImage  = ee.Image(0).byte().paint(trainingPolys_no_RS, 1).paint(evalPolys_no_RS, 2)
polyImage  = polyImage.updateMask(polyImage)
mapidGeoms = polyImage.getMapId({'min': 1, 'max': 2, 'palette': ['red', 'blue']})

map = folium.Map(location=[-1.3621, -45.2738], zoom_start=5)
map = folium.Map()
folium.TileLayer(
    tiles=mapid['tile_fetcher'].url_format,
    attr='Planet',
    overlay=True,
    name='Mosaic composite',
  ).add_to(map)

mapid = supervisedChannel.select(RESPONSE).selfMask().getMapId({'min': 0, 'max': 1, 'pallete':'#ff0000'})

folium.TileLayer(
    tiles=mapid['tile_fetcher'].url_format,
    attr='Google Earth Engine',
    overlay=True,
    name='supervisedLayer',
  ).add_to(map)

folium.TileLayer(
    tiles=mapidGeoms['tile_fetcher'].url_format,
    attr='Google Earth Engine',
    overlay=True,
    name='polygons',
  ).add_to(map)

map.add_child(folium.LayerControl())

map

In [42]:
featureStack = ee.Image.cat([
  image.select(BANDS).unmask(0),
  image.select(RESPONSE).unmask(0)
]).float()

list = ee.List.repeat(1, KERNEL_SIZE)
lists = ee.List.repeat(list, KERNEL_SIZE)
kernel = ee.Kernel.fixed(KERNEL_SIZE, KERNEL_SIZE, lists)

arrays = featureStack.neighborhoodToArray(kernel)

In [43]:
# Convert the feature collections to lists for iteration.
trainingPolysList = trainingPolys_no_RS.toList(trainingPolys_no_RS.size())
evalPolysList = evalPolys_no_RS.toList(evalPolys_no_RS.size())
# These numbers determined experimentally.
n = 20 # Number of shards in each polygon.
N = 200 # Total sample size in each polygon.

#Add some generalism
TRAIN_SIZE = trainingPolys_no_RS.size().getInfo()*N
EVAL_SIZE = evalPolys_no_RS.size().getInfo()*N
print('TRAIN:'+str(TRAIN_SIZE))
print('EVAL:'+str(EVAL_SIZE))

TRAIN:61800
EVAL:14600


In [44]:
trainingPolys = trainingPolys_no_RS
evalPolys = evalPolys_no_RS

# Train/Test Chips Exportation

In [45]:
# Convert the feature collections to lists for iteration.
trainingPolysList = trainingPolys.toList(trainingPolys.size())
evalPolysList = evalPolys.toList(evalPolys.size())
# These numbers determined experimentally.
n = 20 # Number of shards in each polygon.
N = 200 # Total sample size in each polygon.

#Add some generalism
TRAIN_SIZE = trainingPolys.size().getInfo()*N
EVAL_SIZE = evalPolys.size().getInfo()*N
print('TRAIN:'+str(TRAIN_SIZE))
print('EVAL:'+str(EVAL_SIZE))

# Export all the training data (in many pieces), with one task 
# per geometry.
for g in range(trainingPolys.size().getInfo()):
  geomSample = ee.FeatureCollection([])
  for i in range(n):
    sample = arrays.sample(
      region = ee.Feature(trainingPolysList.get(g)).geometry(), 
      scale = SCALE, 
      numPixels = N / n, # Size of the shard.
      seed = i,
      tileScale = 8
    )
    geomSample = geomSample.merge(sample)
  
  desc = TRAINING_BASE + '_g' + str(g)
  task = ee.batch.Export.table.toDrive(
    collection = geomSample,
    description = desc, 
    folder = GDRIVE+'/'+FOLDER_TRAIN, 
    fileNamePrefix = desc,
    fileFormat = 'TFRecord',
    selectors = BANDS + [RESPONSE]
  )
  task.start()

# Export all the evaluation data.
for g in range(evalPolys.size().getInfo()):
  geomSample = ee.FeatureCollection([])
  for i in range(n):
    sample = arrays.sample(
      region = ee.Feature(evalPolysList.get(g)).geometry(), 
      scale = SCALE, 
      numPixels = N / n,
      seed = i,
      tileScale = 8
    )
    geomSample = geomSample.merge(sample)
  
  desc = EVAL_BASE + '_g' + str(g)
  task = ee.batch.Export.table.toDrive(
    collection = geomSample,
    description = desc, 
    folder = GDRIVE+'/'+FOLDER_TEST, 
    fileNamePrefix = desc,
    fileFormat = 'TFRecord',
    selectors = BANDS + [RESPONSE],
  )
  task.start()

TRAIN:61800
EVAL:14600


In [5]:
import numpy as np
from PIL import Image
from matplotlib import pyplot as plt

def parse_tfrecord(example_proto):
  """The parsing function.
  Read a serialized example into the structure defined by FEATURES_DICT.
  Args:
    example_proto: a serialized Example.
  Returns: 
    A dictionary of tensors, keyed by feature name.
  """
  print(FEATURES_DICT)
  return tf.io.parse_single_example(example_proto, FEATURES_DICT)


def to_tuple(inputs):
  """Function to convert a dictionary of tensors to a tuple of (inputs, outputs).
  # Turn the tensors returned by parse_tfrecord into a stack in HWC shape.
  Args:
    inputs: A dictionary of tensors, keyed by feature name.
  Returns: 
    A dtuple of (inputs, outputs).
  """
  inputsList = [inputs.get(key) for key in FEATURES]
  stacked = tf.stack(inputsList, axis=0)
  # Convert from CHW to HWC
  stacked = tf.transpose(stacked, [1, 2, 0])
  return stacked[:,:,:len(BANDS)], stacked[:,:,len(BANDS):]


def get_dataset(pattern):
  """Function to read, parse and format to tuple a set of input tfrecord files.
  Get all the files matching the pattern, parse and convert to tuple.
  Args:
    pattern: A file pattern to match in a Cloud Storage bucket.
  Returns: 
    A tf.data.Dataset
  """
  glob = tf.io.gfile.glob(pattern)
  dataset = tf.data.TFRecordDataset(glob, compression_type='GZIP')
  dataset = dataset.map(parse_tfrecord, num_parallel_calls=5)
  dataset = dataset.map(to_tuple, num_parallel_calls=5)
  return dataset

In [ ]:
import numpy as np
from PIL import Image
from matplotlib import pyplot as plt
def get_training_dataset():
    """Get the preprocessed training dataset
    Returns: 
    A tf.data.Dataset of training data.
    """
    glob = TRAIN_PATH +'v'+SAMPLE_VERSION+'/' + TRAINING_BASE + '*'
    dataset = get_dataset(glob)
    # train_size = dataset.reduce(np.int64(0), lambda x,_ : x + 1).numpy()
    # print(train_size)  # 61370(1 CONTINENTAL NO Rio Grande do Sul), supose to be 61800
    dataset = dataset.shuffle(BUFFER_SIZE, reshuffle_each_iteration=True).batch(BATCH_SIZE).repeat()
    return dataset

training = get_training_dataset()

# print(iter(training.take(1)).next())
print(training.take(1))

In [ ]:
def get_eval_dataset():
    # EVAL_BASE = 'eval_patches_1'
    print(EVAL_BASE)
    glob      = EVAL_PATH + 'v'+SAMPLE_VERSION + '/' + EVAL_BASE + '*'
    dataset   = get_dataset(glob)
    # eval_size = dataset.reduce(np.int64(0), lambda x,_ : x + 1).numpy()
    # print(eval_size)  # 14412 (1 CONTINENTAL)

    dataset = dataset.batch(1).repeat()
    return dataset

evaluation = get_eval_dataset()
print(evaluation.take(1))
# print(iter(evaluation.take(1)).next())

# UNET

In [5]:

from tensorflow.keras import layers
from tensorflow.keras import losses
from tensorflow.keras import models
from tensorflow.keras import metrics
from tensorflow.keras import optimizers


def conv_block(input_tensor, num_filters):
	encoder = layers.Conv2D(num_filters, (3, 3), padding='same')(input_tensor)
	encoder = layers.BatchNormalization()(encoder)
	encoder = layers.Activation('relu')(encoder)
	encoder = layers.Conv2D(num_filters, (3, 3), padding='same')(encoder)
	encoder = layers.BatchNormalization()(encoder)
	encoder = layers.Activation('relu')(encoder)
	return encoder

def encoder_block(input_tensor, num_filters):
	encoder = conv_block(input_tensor, num_filters)
	encoder_pool = layers.MaxPooling2D((2, 2), strides=(2, 2))(encoder)
	return encoder_pool, encoder

def decoder_block(input_tensor, concat_tensor, num_filters):
	decoder = layers.Conv2DTranspose(num_filters, (2, 2), strides=(2, 2), padding='same')(input_tensor)
	decoder = layers.concatenate([concat_tensor, decoder], axis=-1)
	decoder = layers.BatchNormalization()(decoder)
	decoder = layers.Activation('relu')(decoder)
	decoder = layers.Conv2D(num_filters, (3, 3), padding='same')(decoder)
	decoder = layers.BatchNormalization()(decoder)
	decoder = layers.Activation('relu')(decoder)
	decoder = layers.Conv2D(num_filters, (3, 3), padding='same')(decoder)
	decoder = layers.BatchNormalization()(decoder)
	decoder = layers.Activation('relu')(decoder)
	return decoder

def get_model():
	inputs = layers.Input(shape=[None, None, len(BANDS)]) # 256 (shape=[256, 256, len(BANDS)
	encoder0_pool, encoder0 = encoder_block(inputs, 64) # 128
	encoder1_pool, encoder1 = encoder_block(encoder0_pool, 128) # 64
	encoder2_pool, encoder2 = encoder_block(encoder1_pool, 256) # 32
	encoder3_pool, encoder3 = encoder_block(encoder2_pool, 512) # 16
	center = conv_block(encoder3_pool, 1024) # 8 center
	decoder4 = decoder_block(center, encoder3, 512) # 16
	decoder3 = decoder_block(decoder4, encoder2, 256) # 32
	decoder2 = decoder_block(decoder3, encoder1, 128) # 64
	decoder1 = decoder_block(decoder2, encoder0, 64) # 128
	#outputs = layers.Conv2D(1, (1, 1), activation='sigmoid')(decoder1)
	#dropout = tf.keras.layers.Dropout(decoder1, rate=0.5, name="dropout")
	dropout = layers.Dropout(DROPOUT, name="dropout", noise_shape=None, seed=None)(decoder1)
	outputs = layers.Conv2D(1, (1, 1),  activation=tf.nn.sigmoid, padding='same', kernel_initializer=tf.keras.initializers.GlorotNormal())(dropout)
	model = models.Model(inputs=[inputs], outputs=[outputs])


	optimizer = tf.keras.optimizers.Nadam( 0.000005, name='optimizer')

	model.compile(
		optimizer=optimizer, 
		loss=losses.get(LOSS),
		metrics=[metrics.get(metric) for metric in METRICS]
		#options = run_opts
    )

def get_model_512():
	inputs = layers.Input(shape=[None, None, len(BANDS)]) # 256 (shape=[256, 256, len(BANDS)
	encoder0_pool, encoder0 = encoder_block(inputs, 64) # 128
	encoder1_pool, encoder1 = encoder_block(encoder0_pool, 128) # 64
	encoder2_pool, encoder2 = encoder_block(encoder1_pool, 256) # 32
	encoder3_pool, encoder3 = encoder_block(encoder2_pool, 512) # 16
	encoder4_pool, encoder4 = encoder_block(encoder3_pool, 1024) # 16
	center = conv_block(encoder4_pool, 2048) # 8 center
	decoder5 = decoder_block(center, encoder4, 1024) # 16
	decoder4 = decoder_block(decoder5, encoder3, 512) # 16
	decoder3 = decoder_block(decoder4, encoder2, 256) # 32
	decoder2 = decoder_block(decoder3, encoder1, 128) # 64
	decoder1 = decoder_block(decoder2, encoder0, 64) # 128

	dropout = layers.Dropout(DROPOUT, name="dropout", noise_shape=None, seed=None)(decoder1)
	outputs = layers.Conv2D(1, 1,  activation=tf.nn.sigmoid, padding='same', kernel_initializer=tf.keras.initializers.glorot_normal())(dropout)
	model = models.Model( name='unet',inputs=[inputs], outputs=[outputs])
	optimizer = tf.keras.optimizers.Nadam( 0.000005, name='optimizer')

	model.compile(
		optimizer=optimizer, 
		loss=losses.get(LOSS),
		metrics=[metrics.get(metric) for metric in METRICS]
		#options = run_opts
    )

	return model

# Unet with Quantization aware training

https://www.tensorflow.org/model_optimization/guide/quantization/training_example

In [ ]:
from IPython.display import Image
loaded_model = get_model_512()

EPOCH = 0

if EPOCH>0: 
    # DEFAULT
    LOADED_MODEL_DIR = f'{MODEL_DIR}/cp-00{str(EPOCH)}.keras'    
    # IF NEW VERSION IS LOADED FROM SOME WEIGHT FROM PREVIOUR VERSION
    # MODEL_DIR_LAST_WEIGHT_FOR_NEW_VERSION = '/mnt/nfs/assets/Mapbiomas/modelos/mb10-unet-aquaculture/checkpoint/v1_CONTINENTAL'
    # LOADED_MODEL_DIR = f'{MODEL_DIR_LAST_WEIGHT_FOR_NEW_VERSION}/cp-00{str(EPOCH)}.keras'
    print(LOADED_MODEL_DIR)
    loaded_model.load_weights(LOADED_MODEL_DIR)
print(loaded_model.summary())

In [10]:
import numpy as np
from PIL import Image
from matplotlib import pyplot as plt

def previewClass(epoch,log):
    counter = 0
    for batch in evaluation.shuffle(100).take(3):
    # for batch in training.take(3):
        pureImage  = batch[0]
        supervised = batch[1]
        # print(pureImage[0])
        stacked = tf.transpose(pureImage[0], [0, 1, 2]).numpy()
        print(stacked.shape)
        stackedS = tf.transpose(supervised[0], [0, 1, 2]).numpy()
        #tf.contrib.summary.image(str(epoch)+" ->Input/"+str(counter), stacked[0:3], step=0)

        test_pred_raw = loaded_model.predict(pureImage)
        test_pred_raw = tf.transpose(test_pred_raw[0],[0, 1, 2]).numpy()
        fig = plt.figure(figsize=[12,4])
        # show original image
        fig.add_subplot(131)
        # plt.imshow(stacked[:,:,0:3].astype(np.uint8), interpolation='nearest', vmin=0, vmax=255)
        swir1_nir_red = np.stack([stacked[:, :, 3],  # swir1
                          stacked[:, :, 2],  # nir
                          stacked[:, :, 1]], axis=-1)
        plt.imshow(swir1_nir_red.astype(np.uint8), interpolation='nearest', vmin=0, vmax=255)
        fig.add_subplot(132)
        plt.imshow(stackedS[:,:,0], interpolation='nearest',cmap="gray")
        fig.add_subplot(133)
        plt.imshow(test_pred_raw[:,:,0], interpolation='nearest',cmap="gray")
        plt.show()
        #tf.contrib.summary.image(str(epoch)+" -> Out/"+str(counter), test_pred_raw, step=0)
        counter = counter+1
# previewClass(1,1)     

In [ ]:
import matplotlib.pyplot as plt
import math
import datetime, os

plt.style.use("ggplot")
# checkpoint_path = MODEL_DIR+"/cp-{epoch:04d}.ckpt"
# checkpoint_path = LOADED_MODEL_DIR+"/cp-{epoch:04d}.ckpt" # for old keras
checkpoint_path = MODEL_DIR+"/cp-{epoch:04d}.keras"
checkpoint_dir = os.path.dirname(checkpoint_path)
creatDirectory(checkpoint_dir)
print(checkpoint_dir)


tensorboard = tf.keras.callbacks.TensorBoard(log_dir=f'output/v{VERSION}/log_model',write_images=True)
cp_callback  = tf.keras.callbacks.ModelCheckpoint(checkpoint_path,verbose=1, save_weights_only=False,save_best_only=False,save_freq='epoch')
earlyStopping_callback = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=10)
TRAIN_SIZE = 61370
EVAL_SIZE  = 14412


result = loaded_model.fit(x=training,
  epochs=80,
  initial_epoch=0, # REMEMBER TO CHANGE THIS INITIAL EPOCH PARAM, WHEN OTHER MODEL HAS BEEN LOADED
  steps_per_epoch=int(TRAIN_SIZE / BATCH_SIZE),#1000,#int(TRAIN_SIZE / BATCH_SIZE), 
  verbose=1,
  shuffle=True,
  validation_data=evaluation,
  validation_steps=EVAL_SIZE,#1000,
  # callbacks = [cp_callback,img_callback,tensorboard,earlyStopping_callback])
  callbacks = [cp_callback,tensorboard])


# Grid

In [ ]:
# Gather Grid
kernel_buffer   = [512, 512]

ROOT_PATH      = './'
MOSAIC_VERSION = '1'
mosaic_scale   = 10
GRID_IDs       = pygeoj.load(f'{ROOT_PATH}/GRIDS/GRID-ALLCALSSES-COL9.geojson')
GRID           = pygeoj.load(f'{ROOT_PATH}/GRIDS/GRID-ALLCALSSES-COL9-STACK.geojson')


reduced_grid = [int(geo.properties['id']) for geo in GRID if geo.properties['aqua'] == 1]
reduced_grid.sort()
grids_from_mb8 = [1723,1773,1822,2127,2177,2178,2224,2227,2271,2307,2318,2322,2513,838,1672,1720,1823,2128,2256,2270,2272,2273,2274,2308,2517,2517]
full_aqua = reduced_grid+grids_from_mb8
full_aqua_regions = [int(geo.properties['id']) for geo in GRID if geo.properties['id'] in full_aqua]

# Export mosaics to GDrive

In [ ]:
# Export mosaics to GDrive
def doExport(out_image_base,year,region_id, kernel_buffer, roi):
  """Run the image export task."""
  image = ee.Image('projects/'+USER_EE_PROJECT+'/assets/'+USER_PATH+'/mosaic_'+str(year))
  # Export the image, specifying scale and region.
  task = ee.batch.Export.image.toDrive(
    image          = image.select(BANDS).toFloat(),
    description    = out_image_base+'_'+str(year),
    fileNamePrefix = out_image_base+'_'+str(year), 
    folder         = 'mosaics_landsat/'+str(year),
    scale          = mosaic_scale,
    region         = roi,
    fileFormat     = 'GEOTIFF',
    formatOptions  = { 
      'patchDimensions': KERNEL_SHAPE,
      'kernelSize': kernel_buffer,
      'compressed': True,
      'maxFileSize': 157286400
    }
  )
  task.start()  

# Run the export.
for region in full_aqua_regions:
    region_id = int(region.properties['id'])
    image_base_name =f'v{MOSAIC_VERSION}_S2_grid_{region_id}'
    if int(region_id):
      print('Region:',int(region_id))
      for y in range(2016, 2025): 
          doExport(image_base_name,y, kernel_buffer, region.geometry.coordinates[0])

# Prediction

In [7]:
from PIL import Image
import glob,pygeoj
import rasterio
from rasterio.plot import show
import numpy
import matplotlib.pyplot as plt
from osgeo import ogr
from osgeo import gdal
import os
import json
from patchify import patchify, unpatchify
import numpy as np

from datetime import datetime, date, timedelta

def mosaic_predict(mosaic_lzw,year, region_id, output_path, version, kernel_dim, optical_bands, optical_indices, model, mosaic_scale, EPOCH):
    """Executes segmentation over mosaic data exported from EE as .geotiff

    Parameters
    ----------
    mosaic_lzw : geotiff
        Geotiff data representing the LANDSAT mosaic. The region exported must be the same as the region_id
    year : int
        The year of said geotiff
    region_id : int
        The id of the geojson grid that identifies the geotiff region
    output_path: str
        The output path for the segmented data
    version: int
        The segmentation version
    kernel_dim: int
        The dimention of the patches to be segmented
    optical_bands: list
        List of optical bands present on each geotiff
    optical_indices: list
        List of optical indices on each geotiff
    model: Tensorflow model
        The model trained
    mosaic_scale: int
        The scale of the geotiff. Tipically, for landsat the scale is 30
    EPOCH: int
        The epoch of training for the model. For identification

    Returns
    -------
    Geotiff
        A geotiff corresonding to the segmentation fo the target for the mosaic_lzw
    """

    paths_to_check = [
        f'{output_path}/{year}/e{EPOCH}/outimage_v{version}_e{EPOCH}_grid_{region_id}_{year}_lzw.tif',
        f'{output_path}/{year}/e{EPOCH}/outimage_v{version}_e{EPOCH}_grid_{region_id}_{year}_byte_lzw.tif',
    ]
    if check_file_exists(paths_to_check):
        return f'GRID {region_id} jah predito'

    with rasterio.open(mosaic_lzw, 'r') as ds:
        arr = ds.read()


    arr = np.clip(arr, 0, None)
    
    img_arr_original = arr.astype(np.float32)
    
    if img_arr_original.shape[0]> 6: # há banda blue
        img_arr_original = img_arr_original[1:, :, :] # delete blue band

    img_arr_original = np.nan_to_num(img_arr_original, nan=0.0) 

    print(f'ORIGINAL SHAPE: {img_arr_original.shape}')
    
    ''' MAKE THE IMAGE QUADRATIC '''
    arr_shape_xy = np.array(img_arr_original.shape[1:])
    min_dim = arr_shape_xy.min()
    index_min_dim = np.where(arr_shape_xy==min_dim)[0][0]
    if index_min_dim == 0: # [len(bands), x, y], x<y
        img_arr = img_arr_original[:, :, :min_dim] 
    elif index_min_dim ==1:  # [len(bands), x, y], x>y
        img_arr = img_arr_original[:, :min_dim, :]
    
    ''' ENSURE IMAGE DIMENSIONS ARE MULTIPLES OF kernel_dim '''
    
    pad_size = (kernel_dim - (min_dim % kernel_dim)) % kernel_dim  # Compute necessary padding
    
    if pad_size > 0:
        img_arr = np.pad(img_arr, ((0, 0), (0, pad_size), (0, pad_size)), mode='mean') 

    ''' FULL BANDS (6) '''
    disired_dims    = kernel_dim*2 
    bands           = optical_bands + optical_indices
    patches         = patchify(img_arr, (len(bands), disired_dims, disired_dims), step=kernel_dim)
    dim             = patches.shape[1]
    patch2          = patches.reshape((1, dim**2, len(bands), disired_dims, disired_dims))
    patch2_reshaped = patch2[0].reshape((dim**2, len(bands), disired_dims, disired_dims))
    patch3          = np.transpose(patch2_reshaped, [0,2,3,1])

    curr_patch      = tf.data.Dataset.from_tensor_slices(patch3).batch(1)
    with tf.device('/device:GPU:0'):
        predictions_arr = model.predict(curr_patch, batch_size=8,steps=None, verbose=1)
    patchesPerRow  = dim #mixer['patchesPerRow']
    TotalPatches   = dim**2 #mixer['totalPatches']
    patchDimension = [disired_dims,disired_dims] # mixer['patchDimensions']

    counter       = 1
    rowCounter    = 1
    globalCounter = 0
    finalArray    = np.array([])
    rowArray      = np.array([])

    for raw_record in predictions_arr:
        raw_record = np.squeeze(raw_record) #Elimina Bandas
        rows,cols = raw_record.shape
        #print('AQUI o SHAPE',raw_record.shape)
        limite_esquerda = kernel_dim//2               # 128 #limite_esquerda ==> quantidade inicial de COLUNAS
        limite_direita  = kernel_dim + (kernel_dim//2)# 384 #limite_direita ==> quantidade final de COLUNAS
        limite_inferior = kernel_dim + (kernel_dim//2)# 384
        limite_superior = kernel_dim//2               # 128 #limite_superior ==> quantidade inicial de LINHAS
        if rowCounter == 1: # É O PRIMEIRA LINHA
            limite_superior = 0 
        if (counter == 1) or (counter == patchesPerRow+1): # É A PRIMEIRA COLUNA
            limite_esquerda = 0
        if (counter == patchesPerRow+1) and rowCounter == 1: # É A PRIMEIRA COLUNA A PARTIR DA SEGUNDA LINHA
            limite_superior = kernel_dim//2  # 128 (COMEÇAR A TIRAR DA PARTE DE CIMA DO PATCHE)

        if counter == patchesPerRow:  # É O ULTIMA COLUNA
            limite_direita = kernel_dim * 2  # 512 (SELECIONAR BOA PARTE DA DIREITA, SOH EXCLUI 128 DA ESQUERDA)

        if rowCounter == (TotalPatches/patchesPerRow) or (rowCounter == (TotalPatches/patchesPerRow)-1 and counter == patchesPerRow+1):  # É O ULTIMA LINHA
            limite_inferior = kernel_dim * 2 # 512 (SELECIONAR BOA PARTE DO INFERIOR, SOH EXCLUI 128 DO SUPERIOR)
        raw_record = raw_record[limite_superior:limite_inferior,limite_esquerda:limite_direita]
        if rowCounter == 1:
            finalArray = rowArray
        if counter <= patchesPerRow: #PRIMEIRO DA LINHA, ESQUERDA
            if counter == 1:
                rowArray = raw_record
            else:
                rowArray = np.concatenate((rowArray,raw_record), axis = 1)
            counter = counter+1
        else:
            counter = 2
            rowCounter = rowCounter+1
            if np.array_equal(finalArray,rowArray):
                  #print('Concat(ROW)(F)=:'+str(globalCounter))
                finalArray = rowArray
            else:
                  #print('Concat(ROW)=:'+str(globalCounter))
                finalArray = np.concatenate((finalArray,rowArray),axis=0)
            rowArray = raw_record #mod de 73 == 0 ja inicia nova linha
        globalCounter = globalCounter+1
    finalArray = np.concatenate((finalArray,rowArray),axis=0)

    finalArray_padded = dynamic_slice_or_pad(arr_shape_xy, finalArray)
    print(f'finalArray_padded: {finalArray_padded.shape}\n\n')
    
    rows,cols = finalArray_padded.shape
    finalArray_padded = np.array([finalArray_padded]) 

    output_path = f'{output_path}/{year}/e{EPOCH}'
    create_directory(output_path)
    raster_uri     = output_path + '/UNET_v'+version+'grid'+str(region_id)+'_'+str(year)+'.tif'
    try:
        if not np.any(np.isnan(finalArray_padded)):
            finalArray_padded = np.round((finalArray_padded.astype(np.float32))*255).astype(np.uint8)
        raster_uri_lzw = f'{output_path}/outimage_v{version}_e{EPOCH}_grid_'+str(region_id)+'_'+str(year)+'_byte_lzw.tif'
        data_type = "uint8"
    except Exception as inst:
        print(f'ERROR: For grid {region_id} \n {inst.args}, {inst}\n Raster in float')
        raster_uri_lzw = f'{output_path}/outimage_v{version}_e{EPOCH}_grid_'+str(region_id)+'_'+str(year)+'_float_lzw.tif'
        data_type = "float32"  
    

    with rasterio.open(raster_uri,'w',
                  driver="GTiff",
                  height=rows,
                  width=cols,
                  count=1,
                  dtype=data_type,
                  crs='EPSG:4326',#mixer["projection"]["crs"],
                  transform=ds.transform, #mixer["projection"]["affine"]["doubleMatrix"],
                  nodata=0) as dataset:
                      dataset.write(finalArray_padded)
    dataset = gdal.Open(raster_uri, gdal.GA_Update)
    !gdal_translate -of GTiff -ot Byte -co "COMPRESS=LZW" -co "PREDICTOR=2" -co "TILED=YES" {raster_uri} {raster_uri_lzw} 
    !rm {raster_uri}
    print("C'est finiz\n\n")
    return f'GRID {region_id} predito'

In [8]:
def check_file_exists(paths):
    """Verifica se algum dos caminhos especificados já existe."""
    for path in paths:
        if os.path.exists(path):
            return True
    return False
def create_directory(new_folder):
  if not os.path.exists(new_folder):
      print(f'lets make the directory: {new_folder}')
      os.makedirs(new_folder)
  else: return

def dynamic_slice_or_pad(target_shape, predicted_img):
    target_rows, target_cols = target_shape
    pad_rows = target_rows - predicted_img.shape[0]
    pad_cols = target_cols - predicted_img.shape[1]

    # ORIGINAL ROWS IS SMALLER
    if pad_rows < 0:
        predicted_img = predicted_img[:target_rows, :]
    elif pad_rows > 0:
        predicted_img = np.pad(predicted_img, ((0, pad_rows), (0, 0)), mode='constant')

    # ORIGINAL COLUMNS IS SMALLER
    if pad_cols < 0:
        predicted_img = predicted_img[:, :target_cols]
    elif pad_cols > 0:
        predicted_img = np.pad(predicted_img, ((0, 0), (0, pad_cols)), mode='constant')
    
    return predicted_img

In [9]:
import pygeoj
kernel_buffer   = [256, 256]


ROOT_PATH      = '/mnt/nfs/assets/images'
mosaic_scale   = 10
GRID_IDs       = pygeoj.load(f'{ROOT_PATH}/GRIDS/GRID-ALLCALSSES-COL9.geojson')
GRID           = pygeoj.load(f'{ROOT_PATH}/GRIDS/GRID-ALLCALSSES-COL9-STACK.geojson')



reduced_grid = [int(geo.properties['id']) for geo in GRID if geo.properties['aqua'] == 1]
reduced_grid.sort()
print(len(reduced_grid))

569


In [10]:
EPOCH

30

In [11]:
# EPOCH
grids_from_mb8 = [
  1723,
  1773,
  1822,
  2127,
  2177,
  2178,
  2224,
  2227,
  2271,
  2307,
  2318,
  2322,
  2513,
  838,
  1672,
  1720,
  1823,
  2128,
  2256,
  2270,
  2272,
  2273,
  2274,
  2308,
  2517,
  2517
]

In [12]:
print(len(reduced_grid), len(grids_from_mb8))
# grids_from_mb8
# print(reduced_grid+grids_from_mb8)
print(len(reduced_grid+grids_from_mb8))
full_aqua = reduced_grid+grids_from_mb8

569 26
595


In [ ]:
from os import listdir
from os.path import isfile, join
import fnmatch
import re
test_year = 1985
mypath = f'{OUTPUT_PATH}/{test_year}/e64/'#outimage_v{VERSION}*.tif'
full_aqua_ = full_aqua
if test_year ==  2023:
    full_aqua_ = [int(n + 1) for n in full_aqua]
on_folder = [int(number) for file in listdir(mypath) if (isfile(join(mypath, file)) and (fnmatch.fnmatch(file, 'outimage_v*.tif'))) \
             for number in re.findall(r'grid_(\d+)_1985', file)]
on_folder.sort()
print(len(on_folder),'\n\n')
print(on_folder,'\n\n')
# die
# print(reduced_grid, '\n\n')
dif = [number for number in on_folder if number not in full_aqua_]
dif.sort()
print(dif)
print(len(dif))

In [ ]:
import time

start = time.time()
for year in range(2023,2024):
    # full_aqua_ = full_aqua
    if year >= 2016:
        full_aqua_ = [int(n + 1) for n in full_aqua]
    if year == 2023:
      MOSAIC_VERSION = '3'  
    print(f'/n/nYEAR:{year},{MOSAIC_VERSION}')
    i = 0
    add_list=[]
    for region_id in full_aqua_:
        print(f'REGION ID {region_id}')
        tf.keras.backend.clear_session()
        start = time.time()
        
        try:
            MOSAIC_PATH    = f'{ROOT_PATH}/mosaico_sentinel2/{year}'
            search_pattern = os.path.join(MOSAIC_PATH, f'v{MOSAIC_VERSION}_*_S2_grid_{region_id}_{year}*.tif')
            matching_files = glob.glob(search_pattern)
            # print(matching_files)
           
            if matching_files:
                i = i+1
                region_mosaic_file = matching_files[0]
                str_return = mosaic_predict(region_mosaic_file, year, region_id, OUTPUT_PATH, VERSION, KERNEL_SIZE, opticalBands, opticalIndices, loaded_model, mosaic_scale, EPOCH)
                print(str_return)
            else:
                print(i)
                logging.error("No matching:",search_pattern)
                add_list.append(region_id)
                print("No matching:",search_pattern)
        except Exception as e:
            add_list.append(region_id)
            logging.error("The error from mosaic_predict is: ",e)
end = time.time()
print(add_list)
print('Prediction Time per year = '+str(end - start))

In [ ]:
# Image reprojection and warping utility
for year_ in range(2016,2025):
    input_imgs = f'{OUTPUT_PATH}/{year_}/e{EPOCH}/outimage_v{VERSION}*.tif'
    output  = f'{OUTPUT_PATH}/{year_}/e{EPOCH}/{year_}_v{VERSION}_e{EPOCH}.tif'
    !gdalwarp  -r average -multi -wo NUM_THREADS=50 -co TILED=YES -co NUM_THREADS=62 -t_srs EPSG:4326 {input_imgs} {output} -of GTiff -ot Byte  -co COMPRESS=DEFLATE -co PREDICTOR=2 -co ZLEVEL=9  -overwrite -quiet --config GDAL_CACHEMAX 2500 -wm 2500 -co BIGTIFF=YES